# 04. Results Analysis

Loads the fitted models from `models/` (written by `03_train_final`) and
summarizes them: blend weights used, RSF/XGB feature importances, and the
training-cohort blended-score distributions. Optionally computes SHAP
values if the `shap` package is installed.

**This notebook is entirely new** -- the original single notebook had no
comparison-table or SHAP code to move; everything here is fresh, kept
deliberately light since it wasn't part of the original modeling logic.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "liverrisk" / "features.py").exists():
            return p
    raise RuntimeError("Could not locate repo root (liverrisk/features.py not found)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODELS_DIR = REPO_ROOT / "models"
PROCESSED_DIR = REPO_ROOT / "liverrisk" / "data" / "processed"

blend_weights_hep = joblib.load(MODELS_DIR / "blend_weights_hep.joblib")
blend_weights_death = joblib.load(MODELS_DIR / "blend_weights_death.joblib")
train_scores_hep = joblib.load(MODELS_DIR / "train_scores_hepatic.joblib")
train_scores_death = joblib.load(MODELS_DIR / "train_scores_death.joblib")

print("Blend weights (w_cox, w_rsf, w_xgb):")
print("  hepatic:", blend_weights_hep)
print("  death  :", blend_weights_death)

## Training-cohort blended risk-score distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_scores_hep, bins=30)
axes[0].set_title("Blended risk score -- hepatic (train cohort)")
axes[1].hist(train_scores_death, bins=30)
axes[1].set_title("Blended risk score -- death (train cohort)")
plt.tight_layout()
plt.show()

## RSF feature importances

In [ ]:
rsf_hep = joblib.load(MODELS_DIR / "rsf_hep.joblib")
rsf_death = joblib.load(MODELS_DIR / "rsf_death.joblib")

feature_names = joblib.load(MODELS_DIR / "cox_hep.joblib").named_steps["pre"].get_feature_names_out()

def top_importances(rsf_pipeline, feature_names, n=20):
    importances = rsf_pipeline.named_steps["rsf"].feature_importances_
    order = np.argsort(importances)[::-1][:n]
    return pd.Series(importances[order], index=np.asarray(feature_names)[order])

top_importances(rsf_hep, feature_names).iloc[::-1].plot.barh(figsize=(8, 6), title="RSF top-20 feature importances (hepatic)")
plt.tight_layout()
plt.show()

## SHAP (optional)

Only runs if the `shap` package is installed -- it wasn't a dependency of the original notebook, so this is skipped by default rather than failing the notebook.

In [ ]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("shap is not installed -- skipping. `pip install shap` to enable this cell.")

if HAS_SHAP:
    X_hep = pd.read_parquet(PROCESSED_DIR / "hep_X.parquet")
    X_hep_transformed = rsf_hep.named_steps["pre"].transform(X_hep)

    explainer = shap.TreeExplainer(rsf_hep.named_steps["rsf"])
    shap_values = explainer.shap_values(X_hep_transformed[:200])  # subsample for speed
    shap.summary_plot(shap_values, X_hep_transformed[:200], feature_names=feature_names, show=True)